In [1]:
import numpy as np 
import pandas as pd
from sklearn.preprocessing import OneHotEncoder as OHE
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split

In [2]:
titanic = pd.read_csv('titanic.csv')

In [3]:
titanic.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [4]:
titanic["Survived"].sum()

np.int64(342)

In [5]:
titanic.groupby("Sex")["Survived"].mean()

Sex
female    0.742038
male      0.188908
Name: Survived, dtype: float64

In [6]:
titanic.groupby(["Sex", "Pclass"])["Survived"].mean()

Sex     Pclass
female  1         0.968085
        2         0.921053
        3         0.500000
male    1         0.368852
        2         0.157407
        3         0.135447
Name: Survived, dtype: float64

In [7]:
titanic.groupby(["Sex", "Pclass"])["Survived"].aggregate('mean').unstack()

Pclass,1,2,3
Sex,,,
female,0.968085,0.921053,0.500000
male,0.368852,0.157407,0.135447


In [8]:
titanic.pivot_table("Survived", index='Sex', columns='Pclass', aggfunc='mean')

Pclass,1,2,3
Sex,,,
female,0.968085,0.921053,0.500000
male,0.368852,0.157407,0.135447


In [9]:
age = pd.cut(titanic['Age'], [0, 18, 80])
titanic.pivot_table("Survived", ["Sex", age], 'Pclass')

/tmp/nix-shell-762647-3485136323/ipykernel_965755/1899566639.py:2: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  titanic.pivot_table("Survived", ["Sex", age], 'Pclass')


Pclass                  1         2         3
Sex    Age                                   
female (0, 18]   0.909091  1.000000  0.511628
       (18, 80]  0.972973  0.900000  0.423729
male   (0, 18]   0.800000  0.600000  0.215686
       (18, 80]  0.375000  0.071429  0.133663

In [10]:
cat_cols = ["Sex", "Embarked"]
ohe = OHE(sparse_output=False)

encoded = ohe.fit_transform(titanic[cat_cols])

encoded_df = pd.DataFrame(encoded, columns=ohe.get_feature_names_out(cat_cols), index=titanic.index)

titanic = pd.concat([titanic.drop(columns=cat_cols), encoded_df], axis=1)

In [11]:
titanic.head()

,PassengerId,Survived,Pclass,Name,Age,SibSp,Parch,Ticket,Fare,Cabin,Sex_female,Sex_male,Embarked_C,Embarked_Q,Embarked_S,Embarked_nan
0,1,0,3,"Braund, Mr. Owen Harris",22.0,1,0,A/5 21171,7.2500,NaN,0.0,1.0,0.0,0.0,1.0,0.0
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",38.0,1,0,PC 17599,71.2833,C85,1.0,0.0,1.0,0.0,0.0,0.0
2,3,1,3,"Heikkinen, Miss. Laina",26.0,0,0,STON/O2. 3101282,7.9250,NaN,1.0,0.0,0.0,0.0,1.0,0.0
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",35.0,1,0,113803,53.1000,C123,1.0,0.0,0.0,0.0,1.0,0.0
4,5,0,3,"Allen, Mr. William Henry",35.0,0,0,373450,8.0500,NaN,0.0,1.0,0.0,0.0,1.0,0.0


In [12]:
titanic["Cabin_bool"] = titanic["Cabin"].notna().apply(lambda x: 1 if x else 0)
titanic = titanic.drop(columns="Cabin")
titanic.head()

,PassengerId,Survived,Pclass,Name,Age,SibSp,Parch,Ticket,Fare,Sex_female,Sex_male,Embarked_C,Embarked_Q,Embarked_S,Embarked_nan,Cabin_bool
0,1,0,3,"Braund, Mr. Owen Harris",22.0,1,0,A/5 21171,7.2500,0.0,1.0,0.0,0.0,1.0,0.0,0
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",38.0,1,0,PC 17599,71.2833,1.0,0.0,1.0,0.0,0.0,0.0,1
2,3,1,3,"Heikkinen, Miss. Laina",26.0,0,0,STON/O2. 3101282,7.9250,1.0,0.0,0.0,0.0,1.0,0.0,0
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",35.0,1,0,113803,53.1000,1.0,0.0,0.0,0.0,1.0,0.0,1
4,5,0,3,"Allen, Mr. William Henry",35.0,0,0,373450,8.0500,0.0,1.0,0.0,0.0,1.0,0.0,0


In [13]:
titanic.columns

Index(['PassengerId', 'Survived', 'Pclass', 'Name', 'Age', 'SibSp', 'Parch',
       'Ticket', 'Fare', 'Sex_female', 'Sex_male', 'Embarked_C', 'Embarked_Q',
       'Embarked_S', 'Embarked_nan', 'Cabin_bool'],
      dtype='object')

In [14]:
titanic = titanic.drop(columns="PassengerId")
titanic.columns

Index(['Survived', 'Pclass', 'Name', 'Age', 'SibSp', 'Parch', 'Ticket', 'Fare',
       'Sex_female', 'Sex_male', 'Embarked_C', 'Embarked_Q', 'Embarked_S',
       'Embarked_nan', 'Cabin_bool'],
      dtype='object')

In [15]:
titanic["Name"] = titanic["Name"].apply(lambda x: len(x))
titanic["Name"]

0      23
1      51
2      22
3      44
4      24
       ..
886    21
887    28
888    40
889    21
890    19
Name: Name, Length: 891, dtype: int64

In [16]:
titanic['SibSp']

0      1
1      1
2      0
3      1
4      0
      ..
886    0
887    0
888    1
889    0
890    0
Name: SibSp, Length: 891, dtype: int64

In [17]:
titanic["Parch"]

0      0
1      0
2      0
3      0
4      0
      ..
886    0
887    0
888    2
889    0
890    0
Name: Parch, Length: 891, dtype: int64

In [18]:
titanic.rename(columns= {"Name" : "Name_len"}, inplace=True)

In [19]:
titanic["Ticket"] = titanic["Ticket"].apply(lambda x: x.split(" ")[-1])
titanic["Ticket"] = pd.to_numeric(titanic["Ticket"], errors="coerce")
titanic["Ticket"]

0        21171.0
1        17599.0
2      3101282.0
3       113803.0
4       373450.0
         ...    
886     211536.0
887     112053.0
888       6607.0
889     111369.0
890     370376.0
Name: Ticket, Length: 891, dtype: float64

In [20]:
titanic.head()

,Survived,Pclass,Name_len,Age,SibSp,Parch,Ticket,Fare,Sex_female,Sex_male,Embarked_C,Embarked_Q,Embarked_S,Embarked_nan,Cabin_bool
0,0,3,23,22.0,1,0,21171.0,7.2500,0.0,1.0,0.0,0.0,1.0,0.0,0
1,1,1,51,38.0,1,0,17599.0,71.2833,1.0,0.0,1.0,0.0,0.0,0.0,1
2,1,3,22,26.0,0,0,3101282.0,7.9250,1.0,0.0,0.0,0.0,1.0,0.0,0
3,1,1,44,35.0,1,0,113803.0,53.1000,1.0,0.0,0.0,0.0,1.0,0.0,1
4,0,3,24,35.0,0,0,373450.0,8.0500,0.0,1.0,0.0,0.0,1.0,0.0,0


In [21]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
if torch.cuda.is_available(): print("GPU:", torch.cuda.get_device_name(0)) 

Using device: cuda
GPU: NVIDIA GeForce RTX 3050 Laptop GPU


In [22]:
X_df = titanic.drop(columns=["Survived" , "Ticket"])
Y_df = titanic["Survived"]

age_median = X_df["Age"].median()

X_df["Age"] = X_df["Age"].fillna(age_median)

print("NaNs per column:")
print(X_df.isna().sum())

NaNs per column:
Pclass          0
Name_len        0
Age             0
SibSp           0
Parch           0
Fare            0
Sex_female      0
Sex_male        0
Embarked_C      0
Embarked_Q      0
Embarked_S      0
Embarked_nan    0
Cabin_bool      0
dtype: int64


In [23]:
X_df_train, X_df_test, Y_df_train, Y_df_test = train_test_split(X_df, Y_df, test_size=0.2, random_state=42)

In [24]:
X_df_train.dtypes

Pclass            int64
Name_len          int64
Age             float64
SibSp             int64
Parch             int64
Fare            float64
Sex_female      float64
Sex_male        float64
Embarked_C      float64
Embarked_Q      float64
Embarked_S      float64
Embarked_nan    float64
Cabin_bool        int64
dtype: object

In [25]:
X_train = torch.tensor(X_df_train.values, dtype=torch.float32, device = device)
X_test = torch.tensor(X_df_test.values, dtype=torch.float32, device = device)

y_train = torch.tensor(Y_df_train.values, dtype=torch.float32, device = device)
y_test = torch.tensor(Y_df_test.values, dtype=torch.float32, device = device)

y_train = y_train.reshape(-1, 1)
y_test = y_test.reshape(-1, 1)


In [26]:
y_train.shape

torch.Size([712, 1])

In [27]:
X_train.shape

torch.Size([712, 13])

In [28]:
class Titianic(nn.Module):
    
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.hidden_01 = nn.Linear(13, 30)
        self.sigmoid = nn.Sigmoid()
        self.hidden_02 = nn.Linear(30, 10)
        self.output = nn.Linear(10,1)

    def forward(self, x):
        z1 = self.hidden_01(x)
        x1 = self.sigmoid(z1)
        z2 = self.hidden_02(x1)
        x2 = self.sigmoid(z2)
        z3 = self.output(x2)
        x3 = self.sigmoid(z3)
        return x3

In [29]:
model = Titianic()
model.to(device)
model

Titianic(
  (hidden_01): Linear(in_features=13, out_features=30, bias=True)
  (sigmoid): Sigmoid()
  (hidden_02): Linear(in_features=30, out_features=10, bias=True)
  (output): Linear(in_features=10, out_features=1, bias=True)
)

In [30]:
with torch.no_grad():
    pred = model(X_train)

    print("min:", pred.min())
    print("max:", pred.max())
    print("NaNs:", torch.isnan(pred).sum())
    print("Inf:", torch.isinf(pred).sum())

min: tensor(0.5125, device='cuda:0')
max: tensor(0.5273, device='cuda:0')
NaNs: tensor(0, device='cuda:0')
Inf: tensor(0, device='cuda:0')


In [31]:
loss_fn = nn.BCELoss()
optim = torch.optim.Adam(model.parameters(), lr = 0.01)

In [32]:
for epoch in range(1,1001):
    logits = model(X_train)
    loss = loss_fn(logits, y_train)
    optim.zero_grad()
    loss.backward()
    optim.step()
    if epoch % 50 == 0 : print(f"Epoch: {epoch}, Loss: {loss.item()}")


Epoch: 50, Loss: 0.4997592866420746
Epoch: 100, Loss: 0.38836896419525146
Epoch: 150, Loss: 0.3371487855911255
Epoch: 200, Loss: 0.30489224195480347
Epoch: 250, Loss: 0.27695056796073914
Epoch: 300, Loss: 0.24590010941028595
Epoch: 350, Loss: 0.2234290987253189
Epoch: 400, Loss: 0.20554742217063904
Epoch: 450, Loss: 0.19243712723255157
Epoch: 500, Loss: 0.1827850639820099
Epoch: 550, Loss: 0.17484155297279358
Epoch: 600, Loss: 0.169027179479599
Epoch: 650, Loss: 0.16963395476341248
Epoch: 700, Loss: 0.1586960107088089
Epoch: 750, Loss: 0.15293343365192413
Epoch: 800, Loss: 0.14925619959831238
Epoch: 850, Loss: 0.145303875207901
Epoch: 900, Loss: 0.1435474008321762
Epoch: 950, Loss: 0.1380062699317932
Epoch: 1000, Loss: 0.13496705889701843


In [33]:
torch.save(model.state_dict(), "Titanic_model.pth")

In [34]:
model.eval()

with torch.no_grad():
    y_prob = model(X_test)
    y_pred = (y_prob >= 0.5).float()

accuracy = (y_pred == y_test).float().mean()

print("Accuracy:", accuracy.item())

Accuracy: 0.8100558519363403


Since the Accuracy is 81% I would say that the data picked for the model was as a whole impactful.

In [35]:
model.hidden_02.bias

Parameter containing:
tensor([-0.1386, -0.1036, -0.1719,  0.0361, -0.0324, -0.1340,  0.0115, -0.0322,
        -0.0885, -0.1395], device='cuda:0', requires_grad=True)

In [36]:
model.hidden_02.weight

Parameter containing:
tensor([[-0.9396,  1.4530,  0.7588,  0.1309, -1.4639, -0.3118,  0.7753,  1.5427,
         -1.2743,  0.2972, -1.8914,  1.2376,  0.9864, -2.1762,  1.4127, -1.1876,
         -0.9877,  1.1391,  0.9744, -1.5123, -1.1374,  0.8506,  2.9232,  1.9360,
         -1.1112,  0.8059, -1.1421, -1.0264, -0.7314, -1.2273],
        [ 0.9831, -1.6995, -1.3572, -0.0918,  1.5685,  0.5619, -0.8874, -1.7460,
          1.5140, -0.4008,  2.2136, -1.3734, -1.2169,  2.7802, -1.6379,  1.2684,
          1.3971, -1.3851, -0.9783,  2.3255,  1.2272, -1.0983, -3.3240, -2.1219,
          1.3770, -0.7874,  1.2386,  0.9548,  1.2136,  1.4038],
        [ 0.6382, -1.4172, -0.6584,  0.0670,  1.6175,  0.5638, -0.8177, -1.3560,
          1.2408, -0.4681,  1.5779, -0.9753, -0.9704,  2.2391, -1.2742,  1.0155,
          1.1836, -1.2131, -0.9701,  1.9225,  0.9855, -0.8416, -2.9276, -2.0483,
          1.1750, -0.6021,  0.9593,  0.8412,  1.0145,  1.2125],
        [ 0.7628, -1.6188, -0.6930,  0.0305,  1.6342,  0.